# EventGen - entraîner un LSTM plus grand
1. Menu **Exécution ▸ Modifier le type d'exécution ▸ GPU T4**.
2. Lancez les cellules **une par une**, dans l'ordre.
3. À la cellule 1, choisissez le fichier `EventGen_big.zip`.

In [ ]:
# 1. Installation
import torch
assert torch.cuda.is_available(), 'Pas de GPU : activez le GPU T4 puis relancez cette cellule.'
print('GPU :', torch.cuda.get_device_name(0))
from google.colab import files
files.upload()
!unzip -q -o /content/EventGen_big.zip -d /content/
%cd /content/EventGen_big
!pip -q install tabulate

In [ ]:
# 2. Entraînement : 1 à 2 h. Gardez cet onglet ouvert et le PC allumé.
!python scripts/03_train.py --model attention --hid_dim 512 --emb_dim 256 --n_layers 2 --tf 1.0 --batch_size 64 --epochs 30 --patience 4 --max_minutes 100 --tag big

In [ ]:
# 3. Évaluation sur le jeu de test
!python scripts/04_evaluate.py --models attention_big
import pandas as pd
pd.read_csv('results/metrics/comparison.csv').round(3)

In [ ]:
# 4. Exemples de textes
from src import config as C
from src.config import GenerationConfig
from src.inference import EventGenerator
from src.preprocessing import serialize_event
g = EventGenerator(C.MODELS_DIR / 'attention_big', device='cuda')
cfg = GenerationConfig(max_len=60, beam_width=5, no_repeat_ngram=2, length_penalty=1.0, clean_output=True)
for title, types in [('match de foot tunisie contre japon', ['événement sportif']), ('Concert de jazz au jardin', ['concert']),
                     ('Marché de Noël', ['marché']), ("conférence sur l'ia", ['conférence'])]:
    print(title, '->', g.generate_from_text(serialize_event(title, types), 'beam', cfg).text)

In [ ]:
# 5. Téléchargement : environ 400 Mo
# Si le téléchargement ne démarre pas : panneau Fichiers, eventgen_big.zip, clic droit, Télécharger.
!cp results/metrics/comparison.csv results/metrics/comparison_big.csv
!zip -q -r /content/eventgen_big.zip models/attention_big results/metrics/comparison_big.csv
from google.colab import files
files.download('/content/eventgen_big.zip')